# Solution: consume_03 — aggregation & anomaly detection

In [ ]:
from confluent_kafka import Consumer
from collections import defaultdict
from datetime import datetime
from IPython.display import clear_output
import json

THRESHOLDS = {'strom': 50.0, 'wasser': 200.0}

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'aggregation-exercise',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['strom', 'wasser'])

## Step 1 — aggregate

In [ ]:
totals    = defaultdict(lambda: defaultdict(float))
counts    = defaultdict(lambda: defaultdict(int))
mins      = defaultdict(lambda: defaultdict(lambda: float('inf')))
maxs      = defaultdict(lambda: defaultdict(lambda: float('-inf')))
anomalies = []

messages_read, empty_polls = 0, 0
while messages_read < 200 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    try:
        data  = json.loads(msg.value().decode())
        house = data.get('haus', 'unknown')
        topic = msg.topic()
        value = float(data.get('wert', 0.0))

        totals[topic][house] += value
        counts[topic][house] += 1
        mins[topic][house]    = min(mins[topic][house], value)
        maxs[topic][house]    = max(maxs[topic][house], value)

        if value > THRESHOLDS.get(topic, float('inf')):
            anomalies.append({'topic': topic, 'house': house, 'value': value,
                              'offset': msg.offset(), 'partition': msg.partition()})
    except Exception as e:
        print(f'Parse error: {e}')
print(f'Processed {messages_read} events.')

## Step 2 — summary table

In [ ]:
print(f'{"Topic":<7} | {"House":<7} | {"Count":>5} | {"Min":>8} | {"Max":>8} | {"Avg":>8}')
print('-' * 58)
for topic in sorted(totals.keys()):
    unit = 'kWh' if topic == 'strom' else 'L'
    for house in sorted(totals[topic].keys()):
        n   = counts[topic][house]
        avg = totals[topic][house] / n
        print(f'{topic:<7} | {house:<7} | {n:>5} | {mins[topic][house]:>7.1f}{unit} | {maxs[topic][house]:>7.1f}{unit} | {avg:>7.2f}{unit}')

## Step 3 — anomalies

In [ ]:
if not anomalies:
    print('No anomalies. Run the anomaly simulator in exercise_05_produce_stream.')
else:
    print(f'{len(anomalies)} anomalies\n')
    for a in anomalies:
        unit = 'kWh' if a['topic'] == 'strom' else 'L'
        thr  = THRESHOLDS[a['topic']]
        pct  = (a['value'] / thr - 1) * 100
        print(f'{a["topic"]:>6} | {a["house"]:>7} | {a["value"]:>7.1f}{unit} | P{a["partition"]} offset={a["offset"]} ({pct:+.0f}%)')

## Bonus — live aggregation

In [ ]:
totals_live    = defaultdict(lambda: defaultdict(float))
counts_live    = defaultdict(lambda: defaultdict(int))
anomaly_count  = defaultdict(int)
total_received = 0

try:
    while True:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        try:
            data  = json.loads(msg.value().decode())
            house = data.get('haus', 'unknown')
            topic = msg.topic()
            value = float(data.get('wert', 0.0))
            totals_live[topic][house] += value
            counts_live[topic][house] += 1
            total_received            += 1
            if value > THRESHOLDS.get(topic, float('inf')):
                anomaly_count[topic] += 1
        except Exception:
            pass

        if total_received and total_received % 10 == 0:
            clear_output(wait=True)
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'{total_received} events  [{ts}]  anomalies: {dict(anomaly_count)}')
            print(f'{"Topic":<7} | {"House":<7} | {"Count":>5} | {"Avg":>8}')
            print('-' * 38)
            for t in sorted(totals_live.keys()):
                unit = 'kWh' if t == 'strom' else 'L'
                for h in sorted(totals_live[t].keys()):
                    n   = counts_live[t][h]
                    avg = totals_live[t][h] / n
                    print(f'{t:<7} | {h:<7} | {n:>5} | {avg:>7.2f} {unit}')
except KeyboardInterrupt:
    consumer.close()